# Fine-tuning a model on a token classification task

notebook adapted from: https://github.com/huggingface/notebooks/blob/main/examples/token_classification.ipynb


In this notebook, we will fine-tune a BERT model. Specifically, we will apply it to **Named Entity Recognition** as in our previous assignments.

Your task is to understand how we can do this by following the notebook and

1.   understand how we can do this by following the notebook and getting familiar with the HuggingFace ecosystem.
2.   fine-tuning the BERT model with 3 different random seeds and reporting the results in the report.

We will see how to easily load a dataset for this task and use the HuggingFace `Trainer` API to fine-tune a model on it.

Let's install the packages we need from 🤗 : Transformers, Datasets, Seqeval, and Evalaute.

In [1]:
! pip install transformers datasets=='3.6.0' seqeval evaluate

In [2]:
task = "ner"
model_checkpoint = "google-bert/bert-base-uncased"
batch_size = 16

## Loading the dataset

We will use the [🤗 Datasets](https://github.com/huggingface/datasets) and [Evaluate](https://github.com/huggingface/evaluate) libraries to download the data and get the metric we need to use for evaluation (to compare our model to the benchmark). This can be easily done with the functions `load_dataset` and `load`.  

In [3]:
from datasets import load_dataset
from evaluate import load

For our assignment here, we'll use the [CONLL 2003 dataset](https://www.aclweb.org/anthology/W03-0419.pdf).

When prompted with the question to run custom code, you can say "yes".

In [4]:
datasets = load_dataset("conll2003")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

conll2003.py: 0.00B [00:00, ?B/s]

The repository for conll2003 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/conll2003.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

The `datasets` object itself is [`DatasetDict`](https://huggingface.co/docs/datasets/package_reference/main_classes.html#datasetdict), which contains one key for the training, validation and test set.

In [5]:
datasets

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

We can see the training, validation and test sets all have a column for the tokens (the input texts split into words) and one column of labels for each kind of task we introduced before.

To access an actual element, you need to select a split first, then give an index:

In [6]:
datasets["train"][0]

{'id': '0',
 'tokens': ['EU',
  'rejects',
  'German',
  'call',
  'to',
  'boycott',
  'British',
  'lamb',
  '.'],
 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7],
 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0],
 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}

The labels are already coded as integer ids to be easily usable by our model, but the correspondence with the actual categories is stored in the `features` of the dataset:

In [17]:
datasets["train"].features[f"ner_tags"]

Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], id=None), length=-1, id=None)

So for the NER tags, 0 corresponds to 'O', 1 to 'B-PER' etc... On top of the 'O' (which means no special entity), there are four labels for NER here, each prefixed with 'B-' (for beginning) or 'I-' (for intermediate), that indicate if the token is the first one for the current group with the label or not:
- 'PER' for person
- 'ORG' for organization
- 'LOC' for location
- 'MISC' for miscellaneous

Since the labels are lists of `ClassLabel`, the actual names of the labels are nested in the `feature` attribute of the object above:

In [18]:
label_list = datasets["train"].features[f"{task}_tags"].feature.names
label_list

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

To get a sense of what the data looks like, the following function will show some examples picked randomly in the dataset (automatically decoding the labels in passing).

In [19]:
from datasets import ClassLabel, Sequence
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
        elif isinstance(typ, Sequence) and isinstance(typ.feature, ClassLabel):
            df[column] = df[column].transform(lambda x: [typ.feature.names[i] for i in x])
    display(HTML(df.to_html()))

In [20]:
show_random_elements(datasets["train"])

,id,tokens,pos_tags,chunk_tags,ner_tags
0,6672,"[DAC, Dunajska, Streda, 4, 2, 0, 2, 5, 6, 6]","[NNP, NNP, NNP, CD, CD, CD, CD, CD, CD, CD]","[B-NP, I-NP, I-NP, I-NP, I-NP, I-NP, I-NP, I-NP, I-NP, I-NP]","[B-ORG, I-ORG, I-ORG, O, O, O, O, O, O, O]"
1,12561,"[ATLANTA, AT, PITTSBURGH]","[NNP, NNP, NNP]","[B-NP, I-NP, I-NP]","[B-ORG, O, B-LOC]"
2,4508,"[Couples, 73, 68, 72, 72, ,, Craig, Stadler, 73, 72, 67, 73]","[NNS, CD, CD, CD, CD, ,, NNP, NNP, CD, CD, CD, CD]","[B-NP, I-NP, I-NP, I-NP, I-NP, O, B-NP, I-NP, I-NP, I-NP, I-NP, I-NP]","[B-PER, O, O, O, O, O, B-PER, I-PER, O, O, O, O]"
3,9187,"[Hong, Kong, police, regularly, catch, hundreds, of, illegal, immigrants, and, people, who, have, overstayed, their, visas, from, mainland, China, and, send, them, back, .]","[NNP, NNP, NN, RB, VB, NNS, IN, JJ, NNS, CC, NNS, WP, VBP, VBN, PRP$, NNS, IN, JJ, NNP, CC, VB, PRP, RB, .]","[B-NP, I-NP, I-NP, B-ADVP, B-VP, B-NP, B-PP, B-NP, I-NP, O, B-NP, B-NP, B-VP, I-VP, B-NP, I-NP, B-PP, B-NP, I-NP, O, B-VP, B-NP, B-ADVP, O]","[B-LOC, I-LOC, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-LOC, O, O, O, O, O]"
4,8969,"[They, have, already, signed, an, association, agreement, with, the, European, Union, and, are, both, part, of, the, Central, European, Free, Trade, Area, ,, which, also, comprises, Hungary, ,, Slovakia, and, the, Czech, Republic, .]","[PRP, VBP, RB, VBN, DT, NN, NN, IN, DT, NNP, NNP, CC, VBP, DT, NN, IN, DT, JJ, JJ, NNP, NNP, NNP, ,, WDT, RB, VBZ, NNP, ,, NNP, CC, DT, NNP, NNP, .]","[B-NP, B-VP, I-VP, I-VP, B-NP, I-NP, I-NP, B-PP, B-NP, I-NP, I-NP, O, B-VP, B-NP, I-NP, B-PP, B-NP, I-NP, I-NP, I-NP, I-NP, I-NP, O, B-NP, B-ADVP, B-VP, B-NP, O, B-NP, O, B-NP, I-NP, I-NP, O]","[O, O, O, O, O, O, O, O, O, B-ORG, I-ORG, O, O, O, O, O, O, B-ORG, I-ORG, I-ORG, I-ORG, I-ORG, O, O, O, O, B-LOC, O, B-LOC, O, O, B-LOC, I-LOC, O]"
5,377,"[TENNIS, -, RESULTS, AT, TOSHIBA, CLASSIC, .]","[NNS, :, NNP, NNP, NNP, NNP, .]","[B-NP, O, B-NP, I-NP, I-NP, I-NP, O]","[O, O, O, O, B-MISC, I-MISC, O]"
6,11415,"[RE, :, $, 45,020,000]","[NNP, :, $, CD]","[B-NP, O, B-NP, I-NP]","[O, O, O, O]"
7,3334,"[The, French, group, entered, the, German, market, in, 1990, ,, buying]","[DT, JJ, NN, VBD, DT, JJ, NN, IN, CD, ,, VBG]","[B-NP, I-NP, I-NP, B-VP, B-NP, I-NP, I-NP, B-PP, B-NP, O, B-VP]","[O, B-MISC, O, O, O, B-MISC, O, O, O, O, O]"
8,3700,"[Hereford, 1, Doncaster, 0]","[VBN, CD, NNP, CD]","[B-VP, B-NP, I-NP, I-NP]","[B-ORG, O, B-ORG, O]"
9,13824,"[INDICATORS, -, Spain, -, updated, August, 29, .]","[NNS, :, NNP, :, VBN, NNP, CD, .]","[B-NP, O, B-NP, O, B-NP, I-NP, I-NP, O]","[O, O, B-LOC, O, O, O, O, O]"


## Preprocessing the data

Before we can feed those texts to our model, we need to preprocess them. This is done by a 🤗 Transformers `Tokenizer` which will (as the name indicates) tokenize the inputs (including converting the tokens to their corresponding IDs in the pretrained vocabulary) and put it in a format the model expects, as well as generate the other inputs that model requires.

To do all of this, we instantiate our tokenizer with the `AutoTokenizer.from_pretrained` method, which will ensure:

- we get a tokenizer that corresponds to the model architecture we want to use,
- we download the vocabulary used when pretraining this specific checkpoint.

That vocabulary will be cached, so it's not downloaded again the next time we run the cell.

In [21]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

The following assertion ensures that our tokenizer is a fast tokenizers (backed by Rust) from the 🤗 Tokenizers library. Those fast tokenizers are available for almost all models, and we will need some of the special features they have for our preprocessing.

In [22]:
import transformers
assert isinstance(tokenizer, transformers.PreTrainedTokenizerFast)

You can check which type of models have a fast tokenizer available and which don't on the [big table of models](https://huggingface.co/transformers/index.html#bigtable).

You can directly call this tokenizer on one sentence:

In [23]:
tokenizer("Hello, this is one sentence!")

{'input_ids': [101, 7592, 1010, 2023, 2003, 2028, 6251, 999, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

Depending on the model you selected, you will see different keys in the dictionary returned by the cell above. They don't matter much for what we're doing here (just know they are required by the model we will instantiate later), you can learn more about them in [this tutorial](https://huggingface.co/transformers/preprocessing.html) if you're interested.

If, as is the case here, your inputs have already been split into words, you should pass the list of words to your tokenzier with the argument `is_split_into_words=True`:

In [24]:
tokenizer(["Hello", ",", "this", "is", "one", "sentence", "split", "into", "words", "."], is_split_into_words=True)

{'input_ids': [101, 7592, 1010, 2023, 2003, 2028, 6251, 3975, 2046, 2616, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Note that transformers are often pretrained with subword tokenizers, meaning that even if your inputs have been split into words already, each of those words could be split again by the tokenizer. Let's look at an example of that:

In [25]:
example = datasets["train"][4]
print(example["tokens"])

['Germany', "'s", 'representative', 'to', 'the', 'European', 'Union', "'s", 'veterinary', 'committee', 'Werner', 'Zwingmann', 'said', 'on', 'Wednesday', 'consumers', 'should', 'buy', 'sheepmeat', 'from', 'countries', 'other', 'than', 'Britain', 'until', 'the', 'scientific', 'advice', 'was', 'clearer', '.']


In [26]:
tokenized_input = tokenizer(example["tokens"], is_split_into_words=True)
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
print(tokens)

['[CLS]', 'germany', "'", 's', 'representative', 'to', 'the', 'european', 'union', "'", 's', 'veterinary', 'committee', 'werner', 'z', '##wing', '##mann', 'said', 'on', 'wednesday', 'consumers', 'should', 'buy', 'sheep', '##me', '##at', 'from', 'countries', 'other', 'than', 'britain', 'until', 'the', 'scientific', 'advice', 'was', 'clearer', '.', '[SEP]']


Here the words "Zwingmann" and "sheepmeat" have been split in three subtokens.

This means that we need to do some processing on our labels as the input ids returned by the tokenizer are longer than the lists of labels our dataset contain, first because some special tokens might be added (we can a `[CLS]` and a `[SEP]` above) and then because of those possible splits of words in multiple tokens:

In [27]:
len(example[f"{task}_tags"]), len(tokenized_input["input_ids"])

(31, 39)

Thankfully, the tokenizer returns outputs that have a `word_ids` method which can help us.

In [28]:
print(tokenized_input.word_ids())

[None, 0, 1, 1, 2, 3, 4, 5, 6, 7, 7, 8, 9, 10, 11, 11, 11, 12, 13, 14, 15, 16, 17, 18, 18, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, None]


As we can see, it returns a list with the same number of elements as our processed input ids, mapping special tokens to `None` and all other tokens to their respective word. This way, we can align the labels with the processed input ids.

In [29]:
word_ids = tokenized_input.word_ids()
aligned_labels = [-100 if i is None else example[f"{task}_tags"][i] for i in word_ids]
print(len(aligned_labels), len(tokenized_input["input_ids"]))

39 39


Here we set the labels of all special tokens to -100 (the index that is ignored by PyTorch) and the labels of all other tokens to the label of the word they come from. Another strategy is to set the label only on the first token obtained from a given word, and give a label of -100 to the other subtokens from the same word. We propose the two strategies here, just change the value of the following flag:

In [30]:
label_all_tokens = True

We're now ready to write the function that will preprocess our samples. We feed them to the `tokenizer` with the argument `truncation=True` (to truncate texts that are bigger than the maximum size allowed by the model) and `is_split_into_words=True` (as seen above). Then we align the labels with the token ids using the strategy we picked:

In [31]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"{task}_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are automatically
            # ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            # For the other tokens in a word, we set the label to either the current label or -100, depending on
            # the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

This function works with one or several examples. In the case of several examples, the tokenizer will return a list of lists for each key:

In [32]:
tokenize_and_align_labels(datasets['train'][:5])

{'input_ids': [[101, 7327, 19164, 2446, 2655, 2000, 17757, 2329, 12559, 1012, 102], [101, 2848, 13934, 102], [101, 9371, 2727, 1011, 5511, 1011, 2570, 102], [101, 1996, 2647, 3222, 2056, 2006, 9432, 2009, 18335, 2007, 2446, 6040, 2000, 10390, 2000, 18454, 2078, 2329, 12559, 2127, 6529, 5646, 3251, 5506, 11190, 4295, 2064, 2022, 11860, 2000, 8351, 1012, 102], [101, 2762, 1005, 1055, 4387, 2000, 1996, 2647, 2586, 1005, 1055, 15651, 2837, 14121, 1062, 9328, 5804, 2056, 2006, 9317, 10390, 2323, 4965, 8351, 4168, 4017, 2013, 3032, 2060, 2084, 3725, 2127, 1996, 4045, 6040, 2001, 24509, 1012, 102]], 'token_type_ids': [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], 'attention_mask': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1], [1, 1, 1, 1, 1, 1

To apply this function on all the sentences (or pairs of sentences) in our dataset, we just use the `map` method of our `dataset` object we created earlier. This will apply the function on all the elements of all the splits in `dataset`, so our training, validation and testing data will be preprocessed in one single command.

In [33]:
tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Even better, the results are automatically cached by the 🤗 Datasets library to avoid spending time on this step the next time you run your notebook. The 🤗 Datasets library is normally smart enough to detect when the function you pass to map has changed (and thus requires to not use the cache data). For instance, it will properly detect if you change the task in the first cell and rerun the notebook. 🤗 Datasets warns you when it uses cached files, you can pass `load_from_cache_file=False` in the call to `map` to not use the cached files and force the preprocessing to be applied again.

Note that we passed `batched=True` to encode the texts by batches together. This is to leverage the full benefit of the fast tokenizer we loaded earlier, which will use multi-threading to treat the texts in a batch concurrently.

## Fine-tuning the model

Now that our data is ready, we can download the pretrained model and fine-tune it. Since all our tasks are about token classification, we use the `AutoModelForTokenClassification` class. Like with the tokenizer, the `from_pretrained` method will download and cache the model for us. The only thing we have to specify is the number of labels for our problem (which we can get from the features, as seen before):

In [34]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


The warning is telling us we are throwing away some weights (the `vocab_transform` and `vocab_layer_norm` layers) and randomly initializing some other (the `pre_classifier` and `classifier` layers). This is absolutely normal in this case, because we are removing the head used to pretrain the model on a masked language modeling objective and replacing it with a new head for which we don't have pretrained weights, so the library warns us we should fine-tune this model before using it for inference, which is exactly what we are going to do.

To instantiate a `Trainer`, we will need to define three more things. The most important is the [`TrainingArguments`](https://huggingface.co/transformers/main_classes/trainer.html#transformers.TrainingArguments), which is a class that contains all the attributes to customize the training. It requires one folder name, which will be used to save the checkpoints of the model, and all other arguments are optional:

In this assignment, you will experiment with different random seeds and see what the outcome is on the performance of the model. The random seed influences the following steps when training (amongst others):


*  **Weight Initialization:** When we start training, the values for the weights are randomly assigned. This is controlled by the random seed. This impacts the training trajectory in the weight space, e.g. determines which local minimum the model can end up in.
*  **Data Order**: The order in which the model sees the data is also controlled by the random seed. This also influences the training trajectory, determining which data points are e.g. influential in the beginning.


Setting the random seed with a specific numerical value (e.g. 0) makes sure that we can reproduce the results again, introducing some kind of deterministic behavior.

In [35]:
from transformers import set_seed

#first random seed
SEED = 0
set_seed(SEED)

model_name = model_checkpoint.split("/")[-1]
args = TrainingArguments(
    f"{model_name}-finetuned-{task}",
    eval_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=SEED,
    report_to=None,
)

Here we set the evaluation to be done at the end of each epoch, tweak the learning rate, use the `batch_size` defined at the top of the notebook and customize the number of epochs for training, as well as the weight decay.

The last argument to setup everything so we can push the model to the [Hub](https://huggingface.co/models) regularly during training. Remove it if you didn't follow the installation steps at the top of the notebook. If you want to save your model locally in a name that is different than the name of the repository it will be pushed, or if you want to push your model under an organization and not your name space, use the `hub_model_id` argument to set the repo name (it needs to be the full name, including your namespace: for instance `"sgugger/bert-finetuned-ner"` or `"huggingface/bert-finetuned-ner"`).

Then we will need a data collator that will batch our processed examples together while applying padding to make them all the same size (each pad will be padded to the length of its longest example). There is a data collator for this task in the Transformers library, that not only pads the inputs, but also the labels:

In [36]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

The last thing to define for our `Trainer` is how to compute the metrics from the predictions. Here we will load the [`seqeval`](https://github.com/chakki-works/seqeval) metric (which is commonly used to evaluate results on the CONLL dataset) via the Datasets library.

In [37]:
metric = load("seqeval")

This metric takes a list of labels for the predictions and references:

In [38]:
labels = [label_list[i] for i in example[f"{task}_tags"]]
metric.compute(predictions=[labels], references=[labels])

{'LOC': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(2)},
 'ORG': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(1)},
 'PER': {'precision': np.float64(1.0),
  'recall': np.float64(1.0),
  'f1': np.float64(1.0),
  'number': np.int64(1)},
 'overall_precision': np.float64(1.0),
 'overall_recall': np.float64(1.0),
 'overall_f1': np.float64(1.0),
 'overall_accuracy': 1.0}

So we will need to do a bit of post-processing on our predictions:
- select the predicted index (with the maximum logit) for each token
- convert it to its string label
- ignore everywhere we set a label of -100

The following function does all this post-processing on the result of `Trainer.evaluate` (which is a namedtuple containing predictions and labels) before applying the metric:

In [39]:
import numpy as np

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

Note that we drop the precision/recall/f1 computed for each category and only focus on the overall precision/recall/f1/accuracy.

Then we just need to pass all of this along with our datasets to the `Trainer`:

In [40]:
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-3203160651.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


We can now finetune our model by just calling the `train` method:

In [31]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 3


wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.220800,0.062334,0.912548,0.931536,0.921944,0.982144
2,0.048400,0.056808,0.924593,0.939591,0.932031,0.984431
3,0.026000,0.056372,0.933385,0.945184,0.939247,0.985655


TrainOutput(global_step=2634, training_loss=0.07839429088859008, metrics={'train_runtime': 577.4821, 'train_samples_per_second': 72.943, 'train_steps_per_second': 4.561, 'total_flos': 1020668798198106.0, 'train_loss': 0.07839429088859008, 'epoch': 3.0})

The `evaluate` method allows you to evaluate again on the evaluation dataset or on another dataset:

In [32]:
trainer.evaluate()

{'eval_loss': 0.056372202932834625,
 'eval_precision': 0.9333848873177198,
 'eval_recall': 0.9451840250587314,
 'eval_f1': 0.9392474014785172,
 'eval_accuracy': 0.9856545983128664,
 'eval_runtime': 9.9861,
 'eval_samples_per_second': 325.452,
 'eval_steps_per_second': 20.428,
 'epoch': 3.0}

To get the precision/recall/f1 computed for each category now that we have finished training, we can apply the same function as before on the result of the `predict` method:

In [33]:
predictions, labels, _ = trainer.predict(tokenized_datasets["validation"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results = metric.compute(predictions=true_predictions, references=true_labels)
results

{'LOC': {'precision': np.float64(0.9551957831325302),
  'recall': np.float64(0.9690603514132926),
  'f1': np.float64(0.9620781190747062),
  'number': np.int64(2618)},
 'MISC': {'precision': np.float64(0.8185388845247447),
  'recall': np.float64(0.8464662875710804),
  'f1': np.float64(0.8322683706070287),
  'number': np.int64(1231)},
 'ORG': {'precision': np.float64(0.9093959731543624),
  'recall': np.float64(0.9226653696498055),
  'f1': np.float64(0.9159826170931916),
  'number': np.int64(2056)},
 'PER': {'precision': np.float64(0.9789265722752717),
  'recall': np.float64(0.9798945286750165),
  'f1': np.float64(0.9794103113160929),
  'number': np.int64(3034)},
 'overall_precision': np.float64(0.9333848873177198),
 'overall_recall': np.float64(0.9451840250587314),
 'overall_f1': np.float64(0.9392474014785172),
 'overall_accuracy': 0.9856545983128664}

Now that we have trained the model for one seed and obtained the results, let's re-train with **2** more seeds and report the results in your report!

In [38]:
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [40]:
from transformers import set_seed

#second random seed
SEED = 40
set_seed(SEED)

model_name = model_checkpoint.split("/")[-1]
args = TrainingArguments(
    f"{model_name}-finetuned-{task}",
    eval_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=SEED,
    report_to=None,
)
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-2191621262.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [41]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.225800,0.063463,0.920682,0.929746,0.925192,0.982954
2,0.048100,0.058140,0.929794,0.942275,0.935993,0.985194
3,0.024800,0.056768,0.935124,0.946526,0.940791,0.985893


TrainOutput(global_step=2634, training_loss=0.07923998792875692, metrics={'train_runtime': 574.5555, 'train_samples_per_second': 73.314, 'train_steps_per_second': 4.584, 'total_flos': 1022765429074914.0, 'train_loss': 0.07923998792875692, 'epoch': 3.0})

In [42]:
trainer.evaluate()

{'eval_loss': 0.056767579168081284,
 'eval_precision': 0.9351237842617153,
 'eval_recall': 0.9465264570981095,
 'eval_f1': 0.9407905709679213,
 'eval_accuracy': 0.985892894021955,
 'eval_runtime': 11.8402,
 'eval_samples_per_second': 274.488,
 'eval_steps_per_second': 17.229,
 'epoch': 3.0}

In [43]:
predictions, labels, _ = trainer.predict(tokenized_datasets["validation"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results = metric.compute(predictions=true_predictions, references=true_labels)
results

{'LOC': {'precision': np.float64(0.9606358819076457),
  'recall': np.float64(0.9694423223834988),
  'f1': np.float64(0.9650190114068442),
  'number': np.int64(2618)},
 'MISC': {'precision': np.float64(0.8205533596837945),
  'recall': np.float64(0.843216896831844),
  'f1': np.float64(0.8317307692307692),
  'number': np.int64(1231)},
 'ORG': {'precision': np.float64(0.911217183770883),
  'recall': np.float64(0.9285019455252919),
  'f1': np.float64(0.9197783666586365),
  'number': np.int64(2056)},
 'PER': {'precision': np.float64(0.9770190413657256),
  'recall': np.float64(0.980883322346737),
  'f1': np.float64(0.9789473684210527),
  'number': np.int64(3034)},
 'overall_precision': np.float64(0.9351237842617153),
 'overall_recall': np.float64(0.9465264570981095),
 'overall_f1': np.float64(0.9407905709679213),
 'overall_accuracy': 0.985892894021955}

In [41]:
#Seed 3
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=len(label_list))

Some weights of BertForTokenClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [42]:
from transformers import set_seed

# Set random seed!
SEED = 90
set_seed(SEED)

model_name = model_checkpoint.split("/")[-1]
args = TrainingArguments(
    f"{model_name}-finetuned-{task}",
    eval_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    seed=SEED,
    report_to=None,
)
trainer = Trainer(
    model,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/tmp/ipython-input-138518549.py:19: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [43]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.215500,0.058295,0.919661,0.933550,0.926553,0.983208
2,0.046600,0.055002,0.930719,0.945296,0.937951,0.985448


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.215500,0.058295,0.919661,0.933550,0.926553,0.983208
2,0.046600,0.055002,0.930719,0.945296,0.937951,0.985448
3,0.026700,0.056592,0.932664,0.945184,0.938882,0.985766


TrainOutput(global_step=2634, training_loss=0.07577169629417772, metrics={'train_runtime': 535.3779, 'train_samples_per_second': 78.679, 'train_steps_per_second': 4.92, 'total_flos': 1020357467907246.0, 'train_loss': 0.07577169629417772, 'epoch': 3.0})

In [44]:
trainer.evaluate()

{'eval_loss': 0.05659208446741104,
 'eval_precision': 0.9326636494094271,
 'eval_recall': 0.9451840250587314,
 'eval_f1': 0.9388820980108901,
 'eval_accuracy': 0.9857658029771077,
 'eval_runtime': 15.6558,
 'eval_samples_per_second': 207.59,
 'eval_steps_per_second': 13.03,
 'epoch': 3.0}

In [45]:
predictions, labels, _ = trainer.predict(tokenized_datasets["validation"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]
true_labels = [
    [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

results = metric.compute(predictions=true_predictions, references=true_labels)
results

{'LOC': {'precision': np.float64(0.949718574108818),
  'recall': np.float64(0.966768525592055),
  'f1': np.float64(0.9581677077418134),
  'number': np.int64(2618)},
 'MISC': {'precision': np.float64(0.8246031746031746),
  'recall': np.float64(0.8440292445166532),
  'f1': np.float64(0.8342031312725813),
  'number': np.int64(1231)},
 'ORG': {'precision': np.float64(0.9105263157894737),
  'recall': np.float64(0.9255836575875487),
  'f1': np.float64(0.9179932465026531),
  'number': np.int64(2056)},
 'PER': {'precision': np.float64(0.9776609724047306),
  'recall': np.float64(0.980883322346737),
  'f1': np.float64(0.9792694965449161),
  'number': np.int64(3034)},
 'overall_precision': np.float64(0.9326636494094271),
 'overall_recall': np.float64(0.9451840250587314),
 'overall_f1': np.float64(0.9388820980108901),
 'overall_accuracy': 0.9857658029771077}